<a href="https://colab.research.google.com/github/EndritShaqiri/VisionSentry/blob/dex%2Frgb-model-replication/notebooks/train_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train UAV Detector (Thermal or RGB)

Notebook-first workflow for local machines, Colab, or SCC Jupyter. The same notebook can prepare Anti-UAV data for either `ir` or `rgb`, run a smoke test, run a parity training job, and compare metrics against the checked-in thermal weights.


In [1]:
# Replace the URL with your actual repository URL
!git clone --branch dex/rgb-model-replication --single-branch https://github.com/EndritShaqiri/VisionSentry.git
%cd VisionSentry

Cloning into 'VisionSentry'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 100 (delta 41), reused 79 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (100/100), 5.64 MiB | 13.99 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/content/VisionSentry


In [2]:
from pathlib import Path
import os
import sys

try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# Helper to find project root by looking for 'src'
def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'src').exists() and (candidate / 'configs').exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Now attempt the import again
from src.utils.project import find_project_root as verified_find_root

print('Project root:', PROJECT_ROOT)
print('Python:', sys.executable)
print('IN_COLAB =', IN_COLAB)

Project root: /content/VisionSentry
Python: /usr/bin/python3
IN_COLAB = True


In [3]:
import torch

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA visible devices:', os.environ.get('CUDA_VISIBLE_DEVICES'))
print('Device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU 0:', torch.cuda.get_device_name(0))
    _probe = torch.zeros(1, device='cuda')
    print('CUDA probe tensor:', _probe)


Torch: 2.10.0+cu128
CUDA available: True
CUDA visible devices: None
Device count: 1
GPU 0: NVIDIA A100-SXM4-80GB
CUDA probe tensor: tensor([0.], device='cuda:0')


In [4]:
# Toggle these values before running.
MODALITY = 'rgb'  # 'ir' or 'rgb'
RUN_MODE = 'smoke'  # 'smoke' or 'parity'
INSTALL_REQUIREMENTS = True
DOWNLOAD_ANTI_UAV_RGBT = IN_COLAB and MODALITY == 'rgb'
ANTI_UAV_RGBT_URL = 'https://drive.google.com/file/d/1NPYaop35ocVTYWHOYQQHn8YHsM9jmLGr/view'
RUN_THERMAL_BASELINE_VALIDATION = True

RAW_DATA_ROOT = PROJECT_ROOT / 'data' / 'raw'
EXTRACT_ROOT = RAW_DATA_ROOT / 'Anti-UAV-RGBT'
RAW_TRAIN_DIR = RAW_DATA_ROOT / 'train'
RAW_TEST_DIR = None
OUTPUT_ROOT = PROJECT_ROOT / 'data' / ('rgb_uav' if MODALITY == 'rgb' else 'thermal_uav')
DATA_CONFIG = PROJECT_ROOT / 'configs' / ('dataset_rgb_uav.yaml' if MODALITY == 'rgb' else 'dataset_thermal_uav.yaml')
TRAIN_CONFIG = PROJECT_ROOT / 'configs' / ('train_detector_rgb.yaml' if MODALITY == 'rgb' else 'train_detector.yaml')
SPLIT_MANIFEST = RAW_DATA_ROOT / 'train_split_seed42.json'

print('MODALITY =', MODALITY)
print('RUN_MODE =', RUN_MODE)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('DATA_CONFIG =', DATA_CONFIG)
print('TRAIN_CONFIG =', TRAIN_CONFIG)


MODALITY = rgb
RUN_MODE = smoke
OUTPUT_ROOT = /content/VisionSentry/data/rgb_uav
DATA_CONFIG = /content/VisionSentry/configs/dataset_rgb_uav.yaml
TRAIN_CONFIG = /content/VisionSentry/configs/train_detector_rgb.yaml


In [5]:
import shutil
import subprocess
from pathlib import Path

if INSTALL_REQUIREMENTS:
    print('Installing requirements...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / 'requirements.txt')])

if DOWNLOAD_ANTI_UAV_RGBT:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'gdown'])
    RAW_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    archive_path = RAW_DATA_ROOT / 'Anti-UAV-RGBT.zip'

    if not archive_path.exists():
        print(f'Attempting to download dataset to {archive_path}...')
        # Added --remaining-ok to handle potential resume/interruption issues
        try:
            subprocess.check_call([sys.executable, '-m', 'gdown', '--fuzzy', ANTI_UAV_RGBT_URL, '-O', str(archive_path)])
        except subprocess.CalledProcessError as e:
            print(f'Download failed. Please verify the URL: {ANTI_UAV_RGBT_URL}')
            print('If the link is dead, you may need to upload the file manually.')
            raise e

    if archive_path.exists() and not EXTRACT_ROOT.exists():
        print(f'Extracting {archive_path}...')
        EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
        shutil.unpack_archive(str(archive_path), str(EXTRACT_ROOT))

print('Raw data root:', RAW_DATA_ROOT)
print('Extract root:', EXTRACT_ROOT)

Installing requirements...
Attempting to download dataset to /content/VisionSentry/data/raw/Anti-UAV-RGBT.zip...
Extracting /content/VisionSentry/data/raw/Anti-UAV-RGBT.zip...
Raw data root: /content/VisionSentry/data/raw
Extract root: /content/VisionSentry/data/raw/Anti-UAV-RGBT


In [6]:
def find_sequence_root_candidates(root: Path) -> list[Path]:
    if not root.exists():
        return []

    candidates: list[Path] = []
    probe_paths = [root, *root.rglob('*')]
    for path in probe_paths:
        if not path.is_dir():
            continue
        child_dirs = [child for child in path.iterdir() if child.is_dir()]
        if not child_dirs:
            continue
        if any(
            (child / 'IR_label.json').exists()
            or (child / 'RGB_label.json').exists()
            or (child / 'visible_label.json').exists()
            or (child / 'gt' / 'gt.txt').exists()
            for child in child_dirs
        ):
            candidates.append(path)
    return sorted(set(candidates))

# Update RAW_TRAIN_DIR to the specific 'train' folder inside the extraction root
potential_train_dir = EXTRACT_ROOT / 'train'
if potential_train_dir.exists():
    RAW_TRAIN_DIR = potential_train_dir
else:
    train_candidates = find_sequence_root_candidates(EXTRACT_ROOT)
    if len(train_candidates) == 1:
        RAW_TRAIN_DIR = train_candidates[0]
    elif train_candidates:
        print('Discovered candidate train roots:')
        for candidate in train_candidates:
            print(' -', candidate)

print('RAW_TRAIN_DIR =', RAW_TRAIN_DIR)
print('RAW_TEST_DIR =', RAW_TEST_DIR)
print('SPLIT_MANIFEST =', SPLIT_MANIFEST)

RAW_TRAIN_DIR = /content/VisionSentry/data/raw/Anti-UAV-RGBT/train
RAW_TEST_DIR = None
SPLIT_MANIFEST = /content/VisionSentry/data/raw/train_split_seed42.json


In [7]:
import os
from pathlib import Path

def list_dir_recursive(path, depth=2):
    path = Path(path)
    if not path.exists():
        print(f'{path} does not exist')
        return

    print(f'Listing {path}:')
    for root, dirs, files in os.walk(path):
        level = len(Path(root).relative_to(path).parts)
        if level >= depth: continue
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        for f in files[:5]: # show first 5 files
            print(f'{indent}  - {f}')
        if len(files) > 5:
            print(f'{indent}  - ... ({len(files)} total files)')

list_dir_recursive(EXTRACT_ROOT)

Listing /content/VisionSentry/data/raw/Anti-UAV-RGBT:
Anti-UAV-RGBT/
  - framecut.py
  label_new/
    - test.json
    - train.json
    - val.json
  val/
  test/
  train/


In [8]:
import sys
import subprocess

# Ensure we use the correct directory discovered in the previous cell
# If CkWCiPUYqPc- didn't find 'train', we fall back to EXTRACT_ROOT
final_train_dir = RAW_TRAIN_DIR if RAW_TRAIN_DIR.exists() else EXTRACT_ROOT

prepare_cmd = [
    sys.executable, '-m', 'src.utils.prepare_anti_uav',
    '--raw-train-dir', str(final_train_dir),
    '--output-root', str(OUTPUT_ROOT),
    '--modality', MODALITY,
    '--split-manifest', str(SPLIT_MANIFEST),
    '--val-ratio', '0.2',
]
if RAW_TEST_DIR:
    prepare_cmd.extend(['--raw-test-dir', str(RAW_TEST_DIR)])

print('Running:', ' '.join(prepare_cmd))
try:
    subprocess.check_call(prepare_cmd, cwd=PROJECT_ROOT)
except subprocess.CalledProcessError as e:
    print(f'Preparation failed with exit code {e.returncode}. Check if {final_train_dir} contains the expected sequence folders.')
    raise e

check_cmd = [
    sys.executable, '-m', 'src.utils.dataset_checks',
    '--data', str(DATA_CONFIG),
]
print('Running:', ' '.join(check_cmd))
subprocess.check_call(check_cmd, cwd=PROJECT_ROOT)

Running: /usr/bin/python3 -m src.utils.prepare_anti_uav --raw-train-dir /content/VisionSentry/data/raw/Anti-UAV-RGBT/train --output-root /content/VisionSentry/data/rgb_uav --modality rgb --split-manifest /content/VisionSentry/data/raw/train_split_seed42.json --val-ratio 0.2
Running: /usr/bin/python3 -m src.utils.dataset_checks --data /content/VisionSentry/configs/dataset_rgb_uav.yaml


0

In [9]:
from src.detection.train import load_yaml, run_training

run_label = 'rgb_uav' if MODALITY == 'rgb' else 'thermal_uav'
run_name = f"yolo12n_{run_label}_smoke" if RUN_MODE == 'smoke' else f"yolo12n_{run_label}"

TRAIN_CFG = load_yaml(TRAIN_CONFIG)
TRAIN_CFG.update({
    'device': 'auto',
    'name': run_name,
})

if RUN_MODE == 'smoke':
    TRAIN_CFG.update({
        'epochs': 5,
        'imgsz': 640,
        'batch': -1,
        'workers': 2,
        'cache': 'disk',
        'amp': True,
        'verbose': True,
    })
else:
    TRAIN_CFG.update({
        'amp': True,
        'verbose': True,
    })

TRAIN_CFG


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


{'model': 'yolo12n.pt',
 'data': 'configs/dataset_rgb_uav.yaml',
 'imgsz': 640,
 'batch': -1,
 'epochs': 10,
 'device': 'auto',
 'workers': 2,
 'project': 'runs/detect',
 'name': 'yolo12n_rgb_uav_smoke',
 'pretrained': True,
 'optimizer': 'auto',
 'lr0': 0.01,
 'patience': 30,
 'cache': 'disk',
 'exist_ok': False,
 'amp': True,
 'verbose': True}

In [10]:
save_dir = run_training(TRAIN_CFG, project_root=PROJECT_ROOT)
save_dir


Training config:
  amp: True
  batch: -1
  cache: disk
  data: /content/VisionSentry/configs/dataset_rgb_uav.yaml
  device: 0
  epochs: 10
  exist_ok: False
  imgsz: 640
  lr0: 0.01
  model: yolo12n.pt
  name: yolo12n_rgb_uav_smoke
  optimizer: auto
  patience: 30
  pretrained: True
  project: /content/VisionSentry/runs/detect
  verbose: True
  workers: 2
Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/VisionSentry/configs/dataset_rgb_uav.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1

/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:839: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/10      32.9G          0        nan          0          0        640: 100% ━━━━━━━━━━━━ 694/694 1.6s/it 18:39
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 89/89 1.4it/s 1:04
                   all      30289          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, cannot compute metrics without labels


/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:839: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/10      32.9G          0        nan          0          0        640: 100% ━━━━━━━━━━━━ 694/694 1.6s/it 18:27
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 89/89 1.4it/s 1:03
                   all      30289          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, cannot compute metrics without labels


...

/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:839: RuntimeWarning: Mean of empty slice.
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(



10 epochs completed in 3.287 hours.
Optimizer stripped from /content/VisionSentry/runs/detect/yolo12n_rgb_uav_smoke/weights/last.pt, 5.5MB
Optimizer stripped from /content/VisionSentry/runs/detect/yolo12n_rgb_uav_smoke/weights/best.pt, 5.5MB

Validating /content/VisionSentry/runs/detect/yolo12n_rgb_uav_smoke/weights/best.pt...
Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
YOLOv12n summary (fused): 159 layers, 2,556,923 parameters, 0 gradients, 6.3 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 89/89 1.2it/s 1:13


/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:657: RuntimeWarning: Mean of empty slice.
  ax.plot(px, py.mean(1), linewidth=3, color="blue", label=f"all classes {ap[:, 0].mean():.3f} mAP@0.5")
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:703: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:703: RuntimeWarning: Mean of empty slice.
  y = smooth(py.mean(0), 0.1)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/usr/local/lib/python3.12/dist-packages/ultraly

                   all      30289          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, cannot compute metrics without labels
Speed: 0.1ms preprocess, 0.3ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /content/VisionSentry/runs/detect/yolo12n_rgb_uav_smoke

[OK] Training finished. Artifacts saved in: /content/VisionSentry/runs/detect/yolo12n_rgb_uav_smoke
[OK] Best checkpoint (expected): /content/VisionSentry/runs/detect/yolo12n_rgb_uav_smoke/weights/best.pt


PosixPath('/content/VisionSentry/runs/detect/yolo12n_rgb_uav_smoke')

In [11]:
import json
import pandas as pd
from src.detection.validate import run_validation

val_cfg = {
    'weights': str(save_dir / 'weights' / 'best.pt'),
    'data': str(DATA_CONFIG),
    'split': 'val',
    'imgsz': TRAIN_CFG['imgsz'],
    'batch': TRAIN_CFG['batch'],
    'device': 'auto',
    'workers': TRAIN_CFG['workers'],
    'project': 'runs/val',
    'name': f"{TRAIN_CFG['name']}_val",
    'exist_ok': True,
}
metrics_file = run_validation(val_cfg, project_root=PROJECT_ROOT)
print('RGB/IR run metrics:', metrics_file)

comparison_rows = []
current_metrics = json.loads(Path(metrics_file).read_text(encoding='utf-8'))
comparison_rows.append({'model': MODALITY, **current_metrics})

thermal_metrics_file = None
if RUN_THERMAL_BASELINE_VALIDATION:
    thermal_val_cfg = {
        'weights': str(PROJECT_ROOT / 'weights' / 'best.pt'),
        'data': str(PROJECT_ROOT / 'configs' / 'dataset_thermal_uav.yaml'),
        'split': 'val',
        'imgsz': 960,
        'batch': 16,
        'device': 'auto',
        'workers': TRAIN_CFG['workers'],
        'project': 'runs/val',
        'name': 'thermal_checkedin_baseline_val',
        'exist_ok': True,
    }
    thermal_metrics_file = run_validation(thermal_val_cfg, project_root=PROJECT_ROOT)
    thermal_metrics = json.loads(Path(thermal_metrics_file).read_text(encoding='utf-8'))
    comparison_rows.append({'model': 'thermal_baseline', **thermal_metrics})

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


Validation config:
  batch: -1
  data: /content/VisionSentry/configs/dataset_rgb_uav.yaml
  device: 0
  exist_ok: True
  imgsz: 640
  name: yolo12n_rgb_uav_smoke_val
  project: /content/VisionSentry/runs/val
  split: val
  weights: /content/VisionSentry/runs/detect/yolo12n_rgb_uav_smoke/weights/best.pt
  workers: 2
  save_dir: /content/VisionSentry/runs/val/yolo12n_rgb_uav_smoke_val
Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
YOLOv12n summary (fused): 159 layers, 2,556,923 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2084.8±548.4 MB/s, size: 107.0 KB)
val: Scanning /content/VisionSentry/data/rgb_uav/labels/val.cache... 30289 images, 30289 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 30289/30289 7.9Git/s 0.0s
WARNING ⚠️ Labels are missing or empty in /content/VisionSentry/data/rgb_uav/labels/val.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting

ValueError: batch_size should be a positive integer value, but got batch_size=-1

Switch `RUN_MODE` from `smoke` to `parity` after the smoke test succeeds. The parity run keeps the same base model and nominal hyperparameters as the thermal baseline while still using `device='auto'` for Colab convenience.
